<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/05_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5 - Predict 1984-present, and prove the sensors agree

The prediction is one line. The work is proving the 1984-present series measures lakes,
not satellites.

1. Predict the target lakes from `compiledRS` (Collection 1, 1984-2020).
2. Extract 2021-present from Earth Engine (Collection 2), with the ratios and KIVU
   computed **per pixel then reduced**, because the published ratios are medians of
   pixel-wise ratios, not ratios of medians (established in Phase 5's tests).
3. Fit the Collection 1 to Collection 2 handoff on the 2013-2020 overlap, apply it,
   and report how many centimetres of apparent clarity the collection change invents.

In [ ]:
# Cell 1 of 7: repo + Drive + Earth Engine
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee]"], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

EE_PROJECT = "YOUR-EE-PROJECT"  # set to your registered noncommercial Cloud project

In [ ]:
# Cell 2 of 7: predict the Collection 1 record from compiledRS
import joblib
import numpy as np
import pandas as pd
from lakeclarity import config, edi, features, gee, harmonize, locus, predict, viz

viz.use_style()

rf = joblib.load(config.PROCESSED_DIR / "regional_rf.joblib")
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
target_ids = [int(targets["large"]), int(targets["small"])]

# compiledRS is 10.1 GB; stream it and keep only the two target lakes.
c1_path = config.INTERIM_DIR / "compiledRS_targets.parquet"
if not c1_path.exists():
    src = edi.download("compiled_rs")
    edi.stream_filter(src, c1_path, keep_ids=target_ids)

c1 = pd.read_parquet(c1_path)
c1 = features.add_time_columns(c1)
c1 = c1[c1["Pixelcount"] >= config.MIN_PIXELCOUNT]
c1 = c1[~features.has_negative_reflectance(c1)]
c1["secchi_predicted_m"] = 10 ** rf.predict(features.feature_matrix(c1))
print(f"Collection 1: {len(c1):,} usable passes, {c1.year.min()}-{c1.year.max()}")

In [ ]:
# Cell 3 of 7: extract Collection 2 (2013-present) from Earth Engine
# 2013-2020 overlaps compiledRS and is used to fit the handoff; 2021+ is the extension.
gee.initialize(EE_PROJECT)
lakes = locus.load_lakes().set_index("lagoslakeid")

for lake_id in target_ids:
    row = lakes.loc[lake_id]
    geom = gee.lake_geometry(lake_id, row["lake_lat_decdeg"], row["lake_lon_decdeg"])
    fc = gee.extract_lake_timeseries(geom, lake_id, start="2013-01-01", end="2027-01-01")
    task = gee.export_to_drive(fc, description=f"c2_lake_{lake_id}")
    print(f"export started for lake {lake_id}: task {task.id}")

print("\nWait for the Earth Engine tasks to finish (Tasks tab or code.earthengine.google.com),")
print("then re-run from the next cell. The CSVs land in Drive/lake-clarity.")

In [ ]:
# Cell 4 of 7: load the Earth Engine exports, put them in published column names
c2_parts = []
for lake_id in target_ids:
    df = pd.read_csv(config.RAW_DIR.parent / f"c2_lake_{lake_id}.csv")
    df = gee.rename_gee_columns(df)
    df["lagoslakeid"] = lake_id
    c2_parts.append(df)
c2 = pd.concat(c2_parts, ignore_index=True)
c2 = features.add_time_columns(c2)
print(f"Collection 2: {len(c2):,} passes, {c2.year.min()}-{c2.year.max()}")
print("columns present:", sorted(set(config.FEATURES) & set(c2.columns)) == sorted(config.FEATURES))

In [ ]:
# Cell 5 of 7: fit and apply the Collection 1 -> Collection 2 handoff
# Match C1 and C2 passes of the same lake within one day, 2013-2020.
def overlap_frame(c1, c2, ids, tol_days=1):
    parts = []
    for lake_id in ids:
        a = c1[(c1.lagoslakeid == lake_id) & (c1.year.between(2013, 2020))].copy()
        b = c2[(c2.lagoslakeid == lake_id) & (c2.year.between(2013, 2020))].copy()
        a = a.sort_values("sensing_dt"); b = b.sort_values("sensing_dt")
        m = pd.merge_asof(a, b, on="sensing_dt", tolerance=pd.Timedelta(days=tol_days),
                          direction="nearest", suffixes=("_c1", "_c2"))
        parts.append(m.dropna(subset=[f"{config.BANDS[0]}median_c2"]))
    return pd.concat(parts, ignore_index=True)

modelled = [c for c in config.FEATURES]
overlap = overlap_frame(c1, c2, target_ids)
coefs = harmonize.fit_handoff(overlap, columns=modelled)
coefs.to_csv(config.TABLE_DIR / "T12_handoff_coefficients.csv")
print(coefs[["slope", "intercept", "r2", "n"]].round(4).to_string())

c2_2021 = c2[c2.year >= 2021].copy()
c2_corrected = harmonize.apply_handoff(c2_2021, coefs)
c2_corrected["secchi_predicted_m"] = 10 ** rf.predict(features.feature_matrix(c2_corrected))
c2_2021["secchi_predicted_m"] = 10 ** rf.predict(features.feature_matrix(c2_2021))  # uncorrected

In [ ]:
# Cell 6 of 7: how many centimetres does the collection change invent?
shift = harmonize.secchi_shift_from_handoff(rf, c2_2021, c2_corrected)
ceiling = pd.read_csv(config.TABLE_DIR / "T03_variance_ceiling.csv", index_col=0).squeeze()
within_sd_m = float(10 ** ceiling["within_lake_sd_log10"] - 1) * c1["secchi_predicted_m"].median()

print(f"median absolute Secchi shift from the handoff : {shift['abs_median_shift_cm']:.1f} cm")
print(f"95th percentile                               : {shift['p95_abs_shift_cm']:.1f} cm")
print(f"a lake's real interannual 1 sigma             : {100*within_sd_m:.1f} cm")
print("\nIf the shift is a large fraction of the interannual signal, the correction is")
print("not optional. This is the argument for insisting on Earth Engine experience.")

corrected_full = pd.concat([c1, c2_corrected], ignore_index=True)
uncorrected_full = pd.concat([c1, c2_2021], ignore_index=True)
corrected_full.to_parquet(config.PROCESSED_DIR / "predictions_full.parquet", index=False)

In [ ]:
# Cell 7 of 7: figures F21 to F24
for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    passes = corrected_full[corrected_full.lagoslakeid == lake_id]
    print("wrote", viz.save(predict.fig_pass_availability(passes, name), "F21", f"passes_{lake_id}").name)
    print("wrote", viz.save(predict.fig_full_timeseries(
        passes, name, uncorrected=uncorrected_full[uncorrected_full.lagoslakeid == lake_id]),
        "F24", f"timeseries_{lake_id}").name)

print("wrote", viz.save(predict.fig_collection_agreement(coefs, overlap), "F22", "collection_agreement").name)
print("wrote", viz.save(predict.fig_secchi_shift(shift, within_sd_m), "F23", "secchi_shift").name)